In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

# Load .env
load_dotenv(".env")

# Get OpenAI API key
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY was not loaded.")

print("OpenAI API key loaded:", openai_api_key[:10] + "...")

# LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=openai_api_key
)

# Embeddings
embedding = OpenAIEmbeddings(
    api_key=openai_api_key
)

# Load document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(raw_text)

documents = [
    Document(page_content=t)
    for t in texts
]

# FAISS
vectorstore = FAISS.from_texts(
    texts,
    embedding
)

retriever = vectorstore.as_retriever()

print("Base setup complete.")

OpenAI API key loaded: sk-proj-GV...
Base setup complete.


In [8]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

stuff_chain = create_stuff_documents_chain(
    llm,
    prompt,
    document_variable_name="context"
)

response = stuff_chain.invoke({
    "context": documents[:3],
    "question": "What is this document about?"
})

print("\nStuffDocumentsChain Answer:", response)


StuffDocumentsChain Answer: This document provides an overview of LangChain, a framework created by Harrison Chase for building applications with large language models (LLMs). It highlights LangChain's features such as support for retrieval-augmented generation (RAG), agents, memory, and tools, and mentions its common use cases including chatbots, document question answering, and AI workflows.


In [9]:
# Initial prompt to summarize the first chunk
# Use case: Starts with a base answer and refines it using subsequent documents — good when each chunk contributes incrementally.

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt to update the previous summary with new context
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({"context": documents[0].page_content})

# Iteratively refine with remaining docs
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("📄 Refined Summary:\n")
print(summary)

📄 Refined Summary:

LangChain, created by Harrison Chase, is a framework for building applications with large language models (LLMs). It supports features like retrieval-augmented generation (RAG), agents, memory, and tools, and is commonly used in chatbots, document Q&A, and AI workflows.
